In [3]:
import json

with open('/home/as5606/projects/LLaVA-Med/data/instruct/llava_med_instruct_60k_inline_mention.json') as f:
    data = json.load(f)

print(len(data))
print(data[0]['domain'])

56708
{'chest_xray': False, 'mri': False, 'ct_scan': False, 'histology': True, 'gross': False}


## Prototype Based Classifier based with Biomedclip


In [1]:
import json
import torch
import numpy as np
from open_clip import create_model_from_pretrained, get_tokenizer
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

/home/as5606/miniconda3/envs/biomedclip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda"

model, _ = create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224', device=DEVICE
)
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
model.eval()

CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          

In [3]:
# ── Load questions per domain ────────────────────────────────────────────────
DOMAINS = {
    "x_ray":         "/home/as5606/projects/Super_LlaVA_Med/dataset/x_ray/chest_xray_filter_decisions.jsonl",
    "mri":           "/home/as5606/projects/Super_LlaVA_Med/dataset/MRI/mri_filter_decisions.jsonl",
    "histology":     "/home/as5606/projects/Super_LlaVA_Med/dataset/histology/histology_filter_decisions.jsonl",
    "gross_pathology": "/home/as5606/projects/Super_LlaVA_Med/dataset/gross_pathology/gross_filter_decisions.jsonl",
    "ct":            "/home/as5606/projects/Super_LlaVA_Med/dataset/CT/ct_filter_decisions.jsonl",
}

In [4]:
domain_questions = {}
for domain, path in DOMAINS.items():
    qs = [json.loads(l)["question"].replace("<image>", "").strip() for l in open(path)]
    domain_questions[domain] = qs
    print(f"{domain:20s}: {len(qs)} questions")

x_ray               : 20886 questions
mri                 : 45481 questions
histology           : 47076 questions
gross_pathology     : 7021 questions
ct                  : 47568 questions


In [5]:
# ── Encode all questions ──────────────────────────────────────────────────────
def encode_texts(texts, batch_size=256):
    all_feats = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i : i + batch_size]
            tokens = tokenizer(batch).to(DEVICE)
            feats = model.encode_text(tokens)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu())
    return torch.cat(all_feats)

domain_embeddings = {}
for domain, qs in domain_questions.items():
    print(f"Encoding {domain}...")
    domain_embeddings[domain] = encode_texts(qs)

Encoding x_ray...


100%|██████████████████████████████████████████████████████████████████████████████████████| 82/82 [02:27<00:00,  1.80s/it]


Encoding mri...


100%|████████████████████████████████████████████████████████████████████████████████████| 178/178 [05:18<00:00,  1.79s/it]


Encoding histology...


100%|████████████████████████████████████████████████████████████████████████████████████| 184/184 [05:29<00:00,  1.79s/it]


Encoding gross_pathology...


100%|██████████████████████████████████████████████████████████████████████████████████████| 28/28 [00:49<00:00,  1.76s/it]


Encoding ct...


100%|████████████████████████████████████████████████████████████████████████████████████| 186/186 [05:33<00:00,  1.79s/it]


In [6]:
# ── Build prototypes (mean embedding per domain) ──────────────────────────────
prototypes = {}
for domain, embs in domain_embeddings.items():
    proto = embs.mean(dim=0)
    proto = proto / proto.norm()
    prototypes[domain] = proto

domain_names = list(prototypes.keys())
proto_matrix = torch.stack([prototypes[d] for d in domain_names])  # [5, 512]

# ── Evaluate: classify every question by nearest prototype ────────────────────
all_true, all_pred = [], []

for true_domain, embs in domain_embeddings.items():
    sims = embs @ proto_matrix.T          # [N, 5]
    pred_indices = sims.argmax(dim=1)     # [N]
    for idx in pred_indices:
        all_true.append(true_domain)
        all_pred.append(domain_names[idx.item()])

print(classification_report(all_true, all_pred, target_names=domain_names))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(all_true, all_pred, labels=domain_names))


                 precision    recall  f1-score   support

          x_ray       0.49      0.33      0.39     47568
            mri       0.12      0.58      0.20      7021
      histology       0.77      0.57      0.66     47076
gross_pathology       0.54      0.50      0.52     45481
             ct       0.54      0.67      0.60     20886

       accuracy                           0.49    168032
      macro avg       0.49      0.53      0.47    168032
   weighted avg       0.57      0.49      0.52    168032

Confusion matrix (rows=true, cols=pred):
[[13951  1600   797  1553  2985]
 [ 2960 22566  3642  8374  7939]
 [ 1350  6259 27039  7772  4656]
 [  366   563  1169  4097   826]
 [ 7046 10786  2389 11828 15519]]


In [ ]:
# Change the loading to only KEEP decisions
for domain, path in DOMAINS.items():
    qs = [
        json.loads(l)["question"].replace("<image>", "").strip()
        for l in open(path)
        if json.loads(l).get("decision") == "KEEP"
    ]
    domain_questions[domain] = qs
    print(f"{domain:20s}: {len(qs)} questions")